In [13]:
import os
#os.chdir("C:\\Users\\Le Tian\\Desktop\\Ensemble modeling\\comp-401")
os.chdir("/Users/gaoletian/Desktop/Ensemble modeling/comp-401")
os.getcwd()

'/Users/gaoletian/Desktop/Ensemble modeling/comp-401'

In [15]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from multi_omics_integration.processing import load_resize
from multi_omics_integration.func import estimators, estimator_parameters

In [17]:
datasets = {
    "rppa": "../Datasets/kipan/subtyping/RPPA.csv"
}
labels = "../Datasets/kipan/subtyping/Clinical_enc.csv"
target_name = "histological_type_enc"

merged_X, X_modalities, y = load_resize(datasets, labels, target_name)

In [19]:
import scanpy as sc
import pandas as pd

# Load data
rppa = pd.read_csv(datasets['rppa'], index_col=0)

# fill missing values
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
n_rppa = pd.DataFrame(imputer.fit_transform(rppa), columns=rppa.columns)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
rppa_df = pd.DataFrame(scaler.fit_transform(n_rppa), columns=n_rppa.columns)

print("Processed RPPA shape:", rppa_df.shape)

Processed RPPA shape: (736, 202)


In [21]:
import pandas as pd

# Convert y to a pandas Series with same index as rna_processed
y = pd.Series(y, index=rppa_df.index)

# Now reassign
merged_X = rppa_df.loc[y.index]

print(" Merged shape:", merged_X.shape)
print(" Target distribution:\n", y.value_counts())


 Merged shape: (736, 202)
 Target distribution:
 1    467
2    206
0     63
Name: count, dtype: int64


In [23]:
results = []

for name, model in estimators:
    if name not in estimator_parameters:
        print(f"Skipping '{name}' (no param grid defined)")
        continue

    print(f"\nTuning model: {name}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", model)
    ])

    param_grid = {f"clf__{k}": v for k, v in estimator_parameters[name].items()}

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=5,
        scoring="accuracy",
        verbose=1,
        n_jobs=-1
    )

    try:
        grid.fit(merged_X, y)
        results.append({
            "model": name,
            "best_score": grid.best_score_,
            "best_params": grid.best_params_
        })
        print(f" Best score for {name}: {grid.best_score_:.4f}")
        print(f" Best parameters for {name}: {grid.best_params_}")
    except Exception as e:
        print(f" Failed to fit {name}: {e}")


Tuning model: logistic
Fitting 5 folds for each of 5 candidates, totalling 25 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in ve

 Best score for logistic: 0.8519
 Best parameters for logistic: {'clf__C': 100}

Tuning model: random_forest
Fitting 5 folds for each of 16 candidates, totalling 80 fits
 Best score for random_forest: 0.9280
 Best parameters for random_forest: {'clf__max_depth': None, 'clf__n_estimators': 100}

Tuning model: deep_nn
Fitting 5 folds for each of 864 candidates, totalling 4320 fits


/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_

 Best score for deep_nn: 0.9755
 Best parameters for deep_nn: {'clf__activation': 'relu', 'clf__alpha': 0.1, 'clf__hidden_layer_sizes': (50,), 'clf__learning_rate': 'constant', 'clf__solver': 'lbfgs'}
Skipping 'svc' (no param grid defined)


In [24]:
results_df = pd.DataFrame(results).sort_values(by="best_score", ascending=False)
print("\nSummary of Results:")
display(results_df)


Summary of Results:


,model,best_score,best_params
2,deep_nn,0.975538,"{'clf__activation': 'relu', 'clf__alpha': 0.1,..."
1,random_forest,0.928001,"{'clf__max_depth': None, 'clf__n_estimators': ..."
0,logistic,0.851921,{'clf__C': 100}
